# A3 Golden Dataset

Dedicated evaluation workspace: rag_testing/golden_dataset_project.

The original corpus remains read-only in rag_a3. The manifest retains its original
relative source paths, document IDs, hashes, statuses and corpus version.
Paths in config.yaml are resolved relative to that configuration file.
Future outputs belong only in this workspace's golden_dataset directory.

## Environment and corpus diagnostics

Use the **A3 Golden Dataset** kernel from rag_testing/.venv (Python 3.11).
Launch JupyterLab using that environment's Python, with this workspace as the root.
The diagnostic cell reads source files locally only to validate UTF-8 and SHA-256;
it never displays their text or stores their contents in notebook outputs.
No LLM, Ragas, embedding or external API calls are performed.


In [ ]:
import csv
import hashlib
import sys
from collections import Counter
from importlib.metadata import version
from pathlib import Path
import yaml

# Works from the notebook directory, project directory, or testing workspace.
candidates = [Path.cwd(), *Path.cwd().parents]
evaluation_root = next(
    p for base in candidates
    for p in (base, base / "golden_dataset_project")
    if (p / "config.yaml").is_file() and (p / "corpus_manifest.csv").is_file()
).resolve()
expected_env = (evaluation_root.parent / ".venv").resolve()
assert Path(sys.prefix).resolve() == expected_env, "Unexpected Python environment"
assert sys.prefix != sys.base_prefix, "A virtual environment is required"
config = yaml.safe_load((evaluation_root / "config.yaml").read_text(encoding="utf-8"))
source_root = (evaluation_root / config["corpus"]["project_root"]).resolve(strict=True)
output_root = (evaluation_root / config["output"]["directory"]).resolve(strict=True)
assert source_root == Path(r"C:\claude_code_projects\rag_a3").resolve()
assert output_root.is_relative_to(evaluation_root)
assert not output_root.is_relative_to(source_root)
manifest_path = evaluation_root / config["corpus"]["manifest"]
manifest_hash = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
assert manifest_hash == config["corpus"]["manifest_sha256"]
corpus_version = (evaluation_root / config["corpus"]["version_file"]).read_text().strip()
assert corpus_version == config["corpus"]["version"] == "a3-golden-v1-756cab840ac23189"
with manifest_path.open(encoding="utf-8", newline="") as handle:
    rows = list(csv.DictReader(handle))
assert len(rows) == 200
assert {r["document_id"] for r in rows} == {f"G{i:03}" for i in range(1, 201)}
paths, ids, hashes = set(), set(), set()
for row in rows:
    relative = Path(row["source_path"])
    assert not relative.is_absolute() and ".." not in relative.parts
    assert (len(relative.parts) == 3 and relative.parts[0] == "confluence_pages"
            or len(relative.parts) == 3 and relative.parts[:2] == ("local_docs", "other"))
    assert relative.suffix.lower() == ".md"
    raw_path = source_root / relative
    for part in (raw_path, *raw_path.parents):
        if part == source_root:
            break
        assert not (getattr(part.lstat(), "st_file_attributes", 0) & 0x400), "Reparse point rejected"
    path = raw_path.resolve(strict=True)
    assert path.is_relative_to(source_root) and path.is_file()
    path_key = str(path).casefold()
    assert path_key not in paths
    paths.add(path_key)
    source_id = row["page_id"] if row["source_type"] == "confluence" else row["local_id"]
    assert source_id and (row["source_type"], source_id) not in ids
    ids.add((row["source_type"], source_id))
    data = path.read_bytes()
    data.decode("utf-8", errors="strict")
    digest = hashlib.sha256(data).hexdigest()
    assert digest == row["content_sha256"]
    assert digest not in hashes
    hashes.add(digest)
statuses = dict(Counter(r["document_status"] for r in rows))
sources = dict(Counter(r["source_type"] for r in rows))
assert statuses == {"unknown": 174, "possibly_outdated": 26}
assert sources == {"confluence": 194, "manual": 6}
print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Environment:", sys.prefix)
print("Kernel: A3 Golden Dataset (a3-golden-dataset)")
for package in ["jupyterlab", "ipykernel", "pandas", "PyYAML", "markdown-it-py", "nbformat", "nbclient"]:
    print(f"{package}: {version(package)}")
print("Source root:", source_root)
print("Output root:", output_root)
print("Corpus version:", corpus_version)
print("Documents / UTF-8 / SHA-256: 200 / 200 / 200 OK")
print("Unique IDs / paths / source IDs / hashes: OK")
print("Document statuses:", statuses)
print("Sources:", sources)
print("Manifest SHA-256:", manifest_hash)
print("Environment and corpus checks: OK")


In [ ]:

from pathlib import Path
import sys

# Узнаём, какой интерпретатор Python выполняет наш код
print("Интерпретатор Python:")
print(sys.executable)

# Определяем папку, из которой запущен Jupyter
current_folder = Path.cwd()

print("\nТекущая рабочая папка:")
print(current_folder)

# Указываем папку проекта тестирования RAG
project_folder = Path(r"C:\claude_code_projects\rag_testing")

# Проверяем, существует ли наш проект
if project_folder.is_dir():
    print("\n✅ Проект rag_testing найден!")
else:
    print("\n❌ Папка проекта не найдена!")

# Проверяем наличие реестра из 200 документов
manifest_path = (
    project_folder
    / "golden_dataset_project"
    / "corpus_manifest.csv"
)

if manifest_path.is_file():
    print("✅ Реестр документов найден!")
else:
    print("❌ Реестр документов не найден!")


# Указываем путь к реестру документов
# Переменная project_folder создана в предыдущей ячейке

In [ ]:

import pandas as pd


manifest_path = (
    project_folder
    / "golden_dataset_project"
    / "corpus_manifest.csv"
)

# Загружаем CSV-файл в таблицу pandas
# dtype=str позволяет сохранить идентификаторы в текстовом виде
df = pd.read_csv(manifest_path, dtype=str, keep_default_na=False)

# Показываем общее количество документов
print(f"Всего документов: {len(df)}")

# Показываем названия всех столбцов
print("\nСтолбцы реестра:")
for column in df.columns:
    print(f" - {column}")

# Отображаем первые 10 документов в виде таблицы
# Метод display() удобно использовать в Jupyter
display(df.head(10))


# Проверяем распределение документов по бизнес-направлениям

In [ ]:



print("=== ДОКУМЕНТЫ ПО КАТЕГОРИЯМ ===")

# value_counts() считает количество строк для каждой категории
categories = df["primary_category"].value_counts()

display(categories.to_frame(name="Количество документов"))


# Проверяем, какие типы знаний представлены в корпусе

print("\n=== ТИПЫ ЗНАНИЙ ===")

knowledge = df["knowledge_type"].value_counts()

display(knowledge.to_frame(name="Количество документов"))


# Проверяем актуальность документов

print("\n=== СТАТУСЫ ДОКУМЕНТОВ ===")

statuses = df["document_status"].value_counts()

display(statuses.to_frame(name="Количество документов"))


# Проверяем общее количество строк
print(f"\nВсего документов: {len(df)}")


# Создаём счётчик документов по бизнес-сущностям

In [ ]:

import json
from collections import Counter

# Создаём счётчик документов по бизнес-сущностям
entity_counter = Counter()

# Сохраняем ошибки разбора, если они встретятся
parsing_errors = []

# Последовательно обрабатываем каждый документ
for _, row in df.iterrows():

    # Получаем список сущностей из JSON-строки
    try:
        entities = json.loads(row["related_entities"])

        # Проверяем, что получился именно список
        if not isinstance(entities, list):
            raise ValueError("Ожидался список сущностей")

        # Учитываем каждую сущность один раз на документ
        entity_counter.update(set(entities))

    except (json.JSONDecodeError, ValueError, TypeError) as error:
        parsing_errors.append({
            "document_id": row["document_id"],
            "error": str(error)
        })

# Создаём удобную таблицу pandas
df_entities = pd.DataFrame(
    entity_counter.most_common(),
    columns=["Сущность", "Документов"]
)

# Рассчитываем долю документов по каждой сущности
df_entities["Доля, %"] = (
    df_entities["Документов"] / len(df) * 100
).round(1)

print("=== ПОКРЫТИЕ БИЗНЕС-СУЩНОСТЕЙ ===")
display(df_entities)

# Проверяем, удалось ли обработать все записи
print(f"\nОшибок разбора JSON: {len(parsing_errors)}")

if parsing_errors:
    display(pd.DataFrame(parsing_errors))


# Список бизнес-сущностей, которые хотим изучить подробнее

In [ ]:

import json


target_entities = [
    "active_subscription",
    "system_subscription",
    "poseidon"
]

# Превращаем JSON-строки в списки Python
# Копируем таблицу, чтобы не изменять исходный DataFrame
df_check = df.copy()

df_check["entities_list"] = df_check["related_entities"].apply(
    json.loads
)

# Для каждой интересующей сущности находим связанные документы
for entity in target_entities:

    # Оставляем только документы с нужной сущностью
    selected = df_check[
        df_check["entities_list"].apply(
            lambda items: entity in items
        )
    ]

    print(f"\n=== {entity.upper()} ===")
    print(f"Количество документов: {len(selected)}")

    # Отображаем названия, категории и типы знаний
    display(
        selected[
            [
                "document_id",
                "title",
                "primary_category",
                "knowledge_type",
                "document_status"
            ]
        ].reset_index(drop=True)
    )


# Выбираем документ, релевантность которого вызывает вопросы

In [ ]:

from pathlib import Path
import hashlib

# Выбираем документ, релевантность которого вызывает вопросы
document_id = "G087"

# Находим запись в нашем реестре
document = df.loc[
    df["document_id"] == document_id
].iloc[0]

# Определяем путь к исходному документу
source_root = Path(r"C:\claude_code_projects\rag_a3")
file_path = source_root / document["source_path"]

# Проверяем целостность перед чтением
file_bytes = file_path.read_bytes()
actual_hash = hashlib.sha256(file_bytes).hexdigest()

assert actual_hash == document["content_sha256"], (
    "Документ изменился после формирования корпуса!"
)

# Читаем исходный Markdown
content = file_bytes.decode("utf-8")

# Выводим сведения о документе
print("ID:", document_id)
print("Название:", document["title"])
print("Категория:", document["primary_category"])
print("Сущности:", document["related_entities"])

# Ищем упоминания подписок и биллинга
keywords = [
    "активн",
    "подписк",
    "биллинг",
    "уведомлен"
]

# Разбиваем текст на строки
lines = content.splitlines()

# Показываем только строки с интересующими терминами
# и соседние строки для понимания контекста
matches = set()

for i, line in enumerate(lines):
    if any(word in line.lower() for word in keywords):
        matches.update(
            range(max(0, i - 2), min(len(lines), i + 3))
        )

# Ограничиваем вывод первыми 60 строками найденного контекста
for i in sorted(matches)[:60]:
    print(f"{i + 1:4}: {lines[i]}")
